# 01 · Inventura, kvalita a chunkování skutečného korpusu

Tyto notebooky používají skutečné soubory projektu, ne vymyšlené výsledky. Buňky bez dat skončí vysvětlením, které balíčky chybí. Klon repozitáře může vyžadovat přístup; alternativně nahraj jeho kopii do Colab. API klíč pro tuto analýzu nepotřebuješ. Trénování ani nové vyhledávání se bez přepínače nespouští. Rezervované testové otázky se nečtou ani nevyhodnocují.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
# In Colab clone the project; locally reuse the current checkout.
if Path('src/librarian').is_dir():
    ROOT = Path.cwd()
elif Path('ML_Librarian/src/librarian').is_dir():
    ROOT = Path('ML_Librarian').resolve()
else:
    subprocess.run(['git','clone','https://github.com/Kilitar/ML_Librarian.git'],check=True)
    ROOT = Path('ML_Librarian').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src/librarian'))
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
# Set an immutable git ref before starting for exact code-version reproduction.


In [ ]:
# Optional environment setup. Enable in a fresh Colab runtime; CPU is sufficient.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-librarian.txt'],check=True)


In [ ]:
# Operator supplies the ignored data bundle(s), never an API key.
UPLOAD_BUNDLES = False  # In Colab set True and select runtime.local.zip and/or development.local.zip.
if UPLOAD_BUNDLES:
    from google.colab import files
    uploaded = files.upload()
    import io, zipfile
    expected = json.loads((ROOT/'deployment/artifacts.json').read_text())
    for filename, blob in uploaded.items():
        record = expected.get(filename)
        if record is None or hashlib.sha256(blob).hexdigest() != record['sha256']:
            raise ValueError('Unknown/stale bundle: compare with operator manifest; do not bypass SHA check')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            names=z.namelist()
            manifest=json.loads(z.read('bundle_manifest.json'))
            if len(names)!=len(set(names)) or set(names)!=set(manifest)|{'bundle_manifest.json'}:
                raise ValueError('Invalid bundle manifest')
            if sum(i.file_size for i in z.infolist())>40*1024*1024:raise ValueError('Bundle too large')
            staged=[]
            for name, digest in manifest.items():
                if name not in record['files']:raise ValueError('Unexpected member')
                target=(ROOT/name).resolve()
                if not target.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe path')
                content=z.read(name)
                if hashlib.sha256(content).hexdigest()!=digest:raise ValueError('Member SHA mismatch')
                staged.append((target,content))
            for target,content in staged:
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
        print('Verified bundle:',filename)


In [ ]:
from collections import Counter
path=ROOT/'data/normalized/chunks_curated.jsonl'
if not path.exists():
    print('Nejprve nahraj runtime.local.zip a spusť ověřený import výše.')
else:
    rows=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    lengths=sorted(len(r['text']) for r in rows)
    print('Chunks:',len(rows),'sources:',len({r['source_path'] for r in rows}))
    print('Source kinds:',Counter(r.get('source_kind','unknown') for r in rows))
    print('Languages:',Counter(r.get('language','unknown') for r in rows))
    print('Empty:',sum(not r['text'].strip() for r in rows))
    print('Exact duplicate texts:',len(rows)-len({r['text'] for r in rows}))
    print('Length min/median/max:',lengths[0],lengths[len(lengths)//2],lengths[-1])
    from search import corpus_fingerprint
    metadata=json.loads((ROOT/'data/index/multilingual_e5_curated.json').read_text())
    assert corpus_fingerprint(rows)==metadata['corpus_fingerprint']
    print('Ordered corpus fingerprint verified:',metadata['corpus_fingerprint'])


## Skutečná extrakce a chunkování
Následující buňka zobrazí použitý projektový kód; neprovádí ingest z nepřístupného původního kurzu a nespouští jeho notebooky.

In [ ]:
print((ROOT/'src/librarian/ingest.py').read_text(encoding='utf-8'))
print((ROOT/'src/librarian/chunk.py').read_text(encoding='utf-8'))

## Interpretace
Počet pasáží není počet nezávislých faktů. Porovnej duplicity, délky a provenienci; jedna dlouhá pasáž může skrýt odpověď a krátká ztratit nadpis.